# fog_416: GatedPreproc + YOLOv3
Dataset: `datasets/dataset_class.py`. Train function and evaluation: `train.py`. Model: `model/`.

In [1]:
import torch

from config import SEEDS, FOG_TEST_SPLITS
from datasets.dataset_class import FogDataset, make_train_loader, make_test_loaders
from model.spagate import SpaGate
from model.yolo import FogYolo
from train import train_model
from evaluate import evaluate_splits

## Look at the data

In [2]:
train_set = FogDataset("train", use_flip=True) #2/3 foggy, 1/3 clean every time an image is loaded
image, boxes = train_set[0]
print(f"train images: {len(train_set)} | image: {tuple(image.shape)} | boxes: {tuple(boxes.shape)}")

for split in FOG_TEST_SPLITS:
    print(f"{split}: {len(FogDataset(split))} images")

train images: 8111 | image: (3, 416, 416) | boxes: (1, 5)
test_foggy: 2734 images
test_clean: 2734 images
rtts: 4322 images


In [3]:
seed = SEEDS[0]

#train_model builds its own loaders, this cell is only a check that they work
train_loader = make_train_loader(seed)
test_loaders = make_test_loaders()

images, boxes_list = next(iter(train_loader))
print(f"images: {tuple(images.shape)} | boxes in the first image: {tuple(boxes_list[0].shape)}")
print(f"steps/epoch: {len(train_loader)}")
for split, loader in test_loaders.items():
    print(f"{split}: {len(loader)} steps")

images: (32, 3, 416, 416) | boxes in the first image: (1, 5)
steps/epoch: 253
test_foggy: 86 steps
test_clean: 86 steps
rtts: 136 steps


## Build the model

In [4]:
torch.manual_seed(seed)

preproc = SpaGate() 
model = FogYolo(preproc) 
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad) 
print(f"trainable params: {trainable_params:,} | frozen params: {frozen_params:,}")

Overriding model.yaml nc=80 with nc=5
trainable params: 172,416 | frozen params: 103,696,319


## Baseline Metrics

In [6]:
#the frozen yolo behind the untrained preproc, this is the number training has to beat
baseline_metrics = evaluate_splits(model)
print(baseline_metrics.to_string(index=False))

test_foggy | mAP@0.5 0.9206
test_clean | mAP@0.5 0.9396
rtts | mAP@0.5 0.6877
     split    map50  ap50_person  ap50_car  ap50_bus  ap50_bicycle  ap50_motorcycle
test_foggy 0.920593     0.910964  0.881532  0.951360      0.927252         0.931858
test_clean 0.939560     0.926217  0.886734  0.969849      0.961569         0.953430
      rtts 0.687675     0.846427  0.748921  0.544722      0.655843         0.642463


## Train and test

In [5]:
test_metrics = train_model(model, seed, "gated_yolov3") #trains, then reports mAP@0.5 on test_foggy, test_clean and rtts

steps/epoch: 253 | batch_size: 32 | lr: 0.0001
trainable params: 172,416


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(test_metrics.to_string(index=False))

## Test one split by itself

In [ ]:
#for example rtts, optionally after loading the saved preproc weights first (yolo is frozen, so it is not in the checkpoint):
#model.preproc.load_state_dict(torch.load("results/run_gated_yolov3_42/checkpoints/preproc_last.pt", map_location="cuda"))
rtts_metrics = evaluate_splits(model, splits=["rtts"])
print(rtts_metrics.to_string(index=False))